# test-lambda — compare stress suite runs

Loads every `summary.json` under `results/<profile>/` from the repo, builds a side-by-side table, and plots latency, errors, throughput, and GPU peaks. Set the kernel working directory to **`notebook/`** or the **repo root** (the first code cell resolves `results/` automatically).

In [ ]:
%pip install matplotlib -q

from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


def find_results_dir() -> Path:
    cwd = Path.cwd().resolve()
    for root in (cwd, cwd.parent):
        r = root / "results"
        if r.is_dir() and any(r.glob("*/summary.json")):
            return r
    raise FileNotFoundError(
        "No results/ with */summary.json found. cd to repo root or notebook/ and retry."
    )


RESULTS = find_results_dir()
REPO_ROOT = RESULTS.parent
print("RESULTS:", RESULTS)
print("REPO_ROOT:", REPO_ROOT)

In [ ]:
def load_summaries(results_dir: Path) -> list[tuple[str, dict]]:
    rows: list[tuple[str, dict]] = []
    for summary_path in sorted(results_dir.glob("*/summary.json")):
        profile = summary_path.parent.name
        data = json.loads(summary_path.read_text(encoding="utf-8"))
        rows.append((profile, data))
    return rows


def dig(d: dict, *keys, default=None):
    cur = d
    for k in keys:
        if not isinstance(cur, dict) or k not in cur:
            return default
        cur = cur[k]
    return cur


summaries = load_summaries(RESULTS)
if not summaries:
    raise SystemExit("No summary.json files found.")
print("Profiles:", [p for p, _ in summaries])

In [ ]:
def summary_to_row(profile: str, s: dict) -> dict:
    req = s.get("requests") or {}
    cold = s.get("cold_start") or {}
    inv = s.get("gpu_inventory") or {}
    gpus = inv.get("gpus") or []
    gpu_name = gpus[0].get("name") if gpus else None
    peaks = s.get("gpu_peaks") or {}
    first_peak = next(iter(peaks.values()), {}) if peaks else {}
    return {
        "profile": profile,
        "suite_name": s.get("suite_name"),
        "n_total": req.get("n_total"),
        "n_ok": req.get("n_ok"),
        "n_error": req.get("n_error"),
        "error_rate": req.get("error_rate"),
        "ttft_p50_s": dig(req, "ttft_s", "p50"),
        "ttft_p99_s": dig(req, "ttft_s", "p99"),
        "total_p99_s": dig(req, "total_s", "p99"),
        "tpot_p99_s": dig(req, "tpot_s_per_tok", "p99"),
        "tok_per_s_mean": dig(req, "throughput", "output_tokens_per_s_mean_per_request"),
        "cold_mean_ready_s": cold.get("mean_ready_wait_s"),
        "cold_all_ready": cold.get("all_ready"),
        "gpu_name": gpu_name,
        "gpu_mem_max_mib": first_peak.get("mem_used_mib_max"),
        "gpu_util_max_pct": first_peak.get("util_gpu_pct_max"),
    }


compare = pd.DataFrame([summary_to_row(p, s) for p, s in summaries])
compare

## Versus lightest profile (smoke if present, else first row)

Percent deltas use **smoke** as baseline when that folder exists; otherwise alphabetical first profile.

In [ ]:
baseline_name = "smoke" if "smoke" in compare["profile"].values else compare["profile"].iloc[0]
base = compare.loc[compare["profile"] == baseline_name].iloc[0]
print("Baseline profile:", baseline_name)

numeric_cols = [
    "error_rate",
    "ttft_p99_s",
    "total_p99_s",
    "tpot_p99_s",
    "tok_per_s_mean",
    "cold_mean_ready_s",
    "gpu_mem_max_mib",
]

delta_rows = []
for _, row in compare.iterrows():
    out = {"profile": row["profile"]}
    for c in numeric_cols:
        v0, v1 = base[c], row[c]
        if pd.isna(v0) or pd.isna(v1) or v0 == 0:
            out[f"{c}_pct_vs_base"] = None
        else:
            out[f"{c}_pct_vs_base"] = 100.0 * (float(v1) - float(v0)) / float(v0)
    delta_rows.append(out)

deltas = pd.DataFrame(delta_rows)
display(compare[["profile", "n_total", "n_ok", "error_rate", "ttft_p99_s", "total_p99_s", "tpot_p99_s"]])
display(deltas)

## Bar charts (overall request stats from each `summary.json`)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
x = compare["profile"].tolist()
axes[0, 0].bar(x, compare["error_rate"], color="coral")
axes[0, 0].set_title("Error rate")
axes[0, 0].tick_params(axis="x", rotation=25)
axes[0, 1].bar(x, compare["ttft_p99_s"], color="steelblue")
axes[0, 1].set_title("TTFT p99 (s)")
axes[0, 1].tick_params(axis="x", rotation=25)
axes[1, 0].bar(x, compare["total_p99_s"], color="seagreen")
axes[1, 0].set_title("Total latency p99 (s)")
axes[1, 0].tick_params(axis="x", rotation=25)
axes[1, 1].bar(x, compare["tok_per_s_mean"], color="mediumpurple")
axes[1, 1].set_title("Mean output tok/s (per request)")
axes[1, 1].tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

## GPU memory time series (`gpu_timeseries.csv`)

Normalized time from first sample so runs of different lengths align on one axis.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
for profile, _ in summaries:
    csv_path = RESULTS / profile / "gpu_timeseries.csv"
    if not csv_path.is_file():
        continue
    g = pd.read_csv(csv_path)
    if g.empty or "host_unix_ts" not in g.columns or "mem_used_mib" not in g.columns:
        continue
    t0 = g["host_unix_ts"].iloc[0]
    t = g["host_unix_ts"] - t0
    ax.plot(t, g["mem_used_mib"], label=profile, linewidth=1.2)
ax.set_xlabel("Seconds since first sample")
ax.set_ylabel("GPU mem used (MiB)")
ax.set_title("VRAM over time by profile")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Cold start per replica (`cold_start.csv`)

Shows how long each container took to become ready (`ready_ok`).

In [ ]:
cold_frames = []
for profile, _ in summaries:
    p = RESULTS / profile / "cold_start.csv"
    if not p.is_file():
        continue
    df = pd.read_csv(p)
    df.insert(0, "profile", profile)
    cold_frames.append(df)

if cold_frames:
    cold_all = pd.concat(cold_frames, ignore_index=True)
    display(cold_all)
    fig, ax = plt.subplots(figsize=(10, 4))
    for profile in cold_all["profile"].unique():
        sub = cold_all[cold_all["profile"] == profile]
        ax.scatter(sub["container_id"], sub["ready_wait_s"], label=profile, alpha=0.8)
    ax.set_xlabel("container_id")
    ax.set_ylabel("ready_wait_s")
    ax.set_title("Replica cold-start wait by profile")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No cold_start.csv files found.")

## Error mix (`requests.csv`)

Count of `error_class` per profile (excluding successful `ok` rows if you only care about failures).

In [ ]:
req_frames = []
for profile, _ in summaries:
    p = RESULTS / profile / "requests.csv"
    if not p.is_file():
        continue
    df = pd.read_csv(p)
    df.insert(0, "profile", profile)
    req_frames.append(df)

if req_frames:
    all_req = pd.concat(req_frames, ignore_index=True)
    err_mix = (
        all_req.groupby(["profile", "error_class"], dropna=False)
        .size()
        .unstack(fill_value=0)
    )
    display(err_mix)
    err_mix.plot(kind="bar", figsize=(10, 4), rot=0)
    plt.title("Requests by error_class")
    plt.ylabel("count")
    plt.tight_layout()
    plt.show()
else:
    print("No requests.csv files found.")